# 07 - Seed Sensitivity Analysis
Aggregate MLP/LSTM/PatchTST results across multiple seeds and visualize mean ? std.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path('saved_results')
SEEDS = [42, 123, 2024, 7]
METRIC = 'cold_mae'   # overall_mae | warm_mae | cold_mae | overall_cvrmse | overall_wape
DISPLAY_STRATEGIES = [
    '1. FTL', '2. Personalized-FL', '3. Progressive Unfreezing',
    '4. Instance-TL', '5. Fed-SimTL', '6. FedMetaTL'
]

print('Results dir:', RESULTS_DIR.resolve())
print('Seeds:', SEEDS)


In [ ]:
# Load standardized seed summaries
pattern = re.compile(r'^(mlp|lstm|patchtst)_seed_(\d+)_summary\.csv$', re.IGNORECASE)
frames = []
for p in RESULTS_DIR.glob('*_seed_*_summary.csv'):
    m = pattern.match(p.name)
    if not m:
        continue
    model = m.group(1).upper().replace('PATCHTST', 'PatchTST')
    seed = int(m.group(2))
    if seed not in SEEDS:
        continue
    df = pd.read_csv(p)
    if 'model' not in df.columns:
        df['model'] = model
    if 'seed' not in df.columns:
        df['seed'] = seed
    frames.append(df)

if not frames:
    raise FileNotFoundError('No standardized seed summary CSVs found. Run export cells in 03/04/06 first.')

all_seed_results = pd.concat(frames, ignore_index=True)
all_seed_results['model'] = all_seed_results['model'].replace({'MLP':'MLP','LSTM':'LSTM','PATCHTST':'PatchTST'})
all_seed_results = all_seed_results.sort_values(['model','strategy','seed']).reset_index(drop=True)
print('Loaded rows:', len(all_seed_results))
display(all_seed_results.head(20))


In [ ]:
# Aggregate mean/std over seeds
required_cols = {'model','seed','strategy',METRIC}
missing = [c for c in required_cols if c not in all_seed_results.columns]
if missing:
    raise ValueError(f'Missing columns for aggregation: {missing}')

agg = (all_seed_results
       .groupby(['model','strategy'], as_index=False)[METRIC]
       .agg(['mean','std','count'])
       .reset_index())
agg.columns = ['model','strategy',f'{METRIC}_mean',f'{METRIC}_std','n_seeds']

agg = agg[agg['strategy'].isin(DISPLAY_STRATEGIES)]
agg = agg.sort_values(['model','strategy']).reset_index(drop=True)
display(agg)
agg.to_csv(RESULTS_DIR / f'seed_agg_{METRIC}.csv', index=False)
print('Saved:', RESULTS_DIR / f'seed_agg_{METRIC}.csv')


In [ ]:
# Plot 1: per-model bars over strategies (mean ? std across seeds)
models = ['MLP','LSTM','PatchTST']
for model in models:
    d = agg[agg['model'] == model].copy()
    if d.empty:
        continue
    d['strategy'] = pd.Categorical(d['strategy'], categories=DISPLAY_STRATEGIES, ordered=True)
    d = d.sort_values('strategy')

    plt.figure(figsize=(11, 4.8))
    x = np.arange(len(d))
    y = d[f'{METRIC}_mean'].values
    e = np.nan_to_num(d[f'{METRIC}_std'].values, nan=0.0)
    plt.bar(x, y, yerr=e, capsize=6, color='#ef746c', edgecolor='#ef746c', alpha=0.95)

    for xi, yi in zip(x, y):
        plt.text(xi, yi, f'{yi:.3f}', ha='center', va='bottom', fontsize=10, color='#e85f57', fontweight='bold')

    plt.xticks(x, d['strategy'], rotation=22, ha='right')
    plt.ylabel(METRIC.replace('_', ' ').title())
    plt.title(f'{model}: {METRIC} across strategies (mean ? std over seeds)')
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot 2: seed-level bars for one selected strategy across models
FOCUS_STRATEGY = '1. FTL'
d = all_seed_results[all_seed_results['strategy'] == FOCUS_STRATEGY].copy()
if d.empty:
    print(f'No rows for strategy: {FOCUS_STRATEGY}')
else:
    d = d[d['seed'].isin(SEEDS)].sort_values(['seed','model'])
    pivot = d.pivot_table(index='seed', columns='model', values=METRIC, aggfunc='mean')
    pivot = pivot[['MLP','LSTM','PatchTST']] if set(['MLP','LSTM','PatchTST']).issubset(pivot.columns) else pivot

    ax = pivot.plot(kind='bar', figsize=(10.5, 4.8), color=['#ef746c','#69a2ff','#6bbf8f'])
    plt.title(f'Seed sensitivity for {FOCUS_STRATEGY} ({METRIC})')
    plt.xlabel('Seed')
    plt.ylabel(METRIC.replace('_', ' ').title())
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
    display(pivot.reset_index())


## How To Run
1. In each model notebook, set `SEED` (e.g. `SEED=42`), run training/evaluation, then run the final standardized export cell.
2. Repeat for all seeds in `SEEDS`.
3. Run this notebook to aggregate and plot.
